# 06 - Tableau de Bord de Decision

Objectif de la phase 6:
- Generer automatiquement un rapport textuel pour chaque patient.
- Produire une vue tableau de bord des cas et priorites.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.decision.engine import generer_recommandation
from src.decision.rules import appliquer_regle_securite_negatif
from src.decision.triage import appliquer_triage
from src.reporting.report_generator import generer_rapports_batch, creer_tableau_bord

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
RNG = np.random.default_rng(123)
print("Imports OK")

Imports OK


## 1) Creation d un lot de predictions (20 patients)

In [2]:
n_patients = 20
probas = RNG.dirichlet(alpha=[1.6, 1.3, 1.5, 1.2], size=n_patients)

# Injecter quelques cas 'notumor' proches du seuil pour tester la securite
for idx in [3, 9, 15]:
    probas[idx] = np.array([0.02, 0.03, 0.92, 0.03])

patient_ids = [f"P_{i+1:05d}" for i in range(n_patients)]
probas[:3]

array([[0.07241632, 0.46361312, 0.3994472 , 0.06452336],
       [0.13156327, 0.1486822 , 0.41674768, 0.30300685],
       [0.49507931, 0.08795456, 0.00480707, 0.41215905]])

## 2) Pipeline SAD -> decision + securite + triage

In [3]:
decisions = []
for pid, p in zip(patient_ids, probas):
    d = generer_recommandation(p, CLASSES, patient_id=pid)
    d = appliquer_regle_securite_negatif(d)
    d = appliquer_triage(d)
    decisions.append(d)

len(decisions)

20

## 3) Generation des 20 rapports textuels

In [4]:
rapports = generer_rapports_batch(decisions)
print(f"Rapports generes: {len(rapports)}")

# Afficher les 2 premiers exemples
print(rapports[0])
print()
print(rapports[1])

Rapports generes: 20
RAPPORT D'AIDE A LA DECISION
Patient ID: P_00001 Date: 05/03/2026

PREDICTION PRINCIPALE
---
Classe: Meningiome
Confiance: 46.4%
Niveau de certitude: TRES_FAIBLE [A_VERIFIER]

SCORES PAR CLASSE
---
- Meningiome: 46.4%
- Pas de tumeur: 39.9%
- Gliome: 7.2%
- Tumeur pituitaire: 6.5%

RECOMMANDATIONS CLINIQUES
---
Diagnostic: Incertitude elevee
Action: Double lecture obligatoire + IRM complementaire
Priorite: [~] Elevée (24h)
Revision humaine: Obligatoire

ELEMENTS D'ATTENTION
---
- Cas incertain: relecture senior prioritaire

RAPPORT D'AIDE A LA DECISION
Patient ID: P_00002 Date: 05/03/2026

PREDICTION PRINCIPALE
---
Classe: Pas de tumeur
Confiance: 41.7%
Niveau de certitude: TRES_FAIBLE [A_VERIFIER]

SCORES PAR CLASSE
---
- Pas de tumeur: 41.7%
- Tumeur pituitaire: 30.3%
- Meningiome: 14.9%
- Gliome: 13.2%

RECOMMANDATIONS CLINIQUES
---
Diagnostic: Prédiction 'Pas de tumeur' à 41.7% confiance - Seuil de sécurité non atteint
Action: Verification obligatoire (risque f

## 4) Tableau de bord operationnel

In [5]:
df_dashboard = creer_tableau_bord(decisions)
display(df_dashboard.head(20))

print("")
print("Repartition des priorites:")
print(df_dashboard['priorite'].value_counts())

print("")
print("Alertes securite:", int(df_dashboard['alerte_securite'].sum()))

,patient_id,classe_predite,confiance,niveau_confiance,priorite,revision_requise,alerte_securite
10,P_00011,Meningiome,0.706134,MOYENNE,Elevée (24h),True,False
6,P_00007,Gliome,0.690160,MOYENNE,Elevée (24h),True,False
12,P_00013,Meningiome,0.472198,TRES_FAIBLE,Elevée (24h),True,False
0,P_00001,Meningiome,0.463613,TRES_FAIBLE,Elevée (24h),True,False
5,P_00006,Meningiome,0.463265,TRES_FAIBLE,Elevée (24h),True,False
17,P_00018,Meningiome,0.433979,TRES_FAIBLE,Elevée (24h),True,False
13,P_00014,Tumeur pituitaire,0.415377,TRES_FAIBLE,Elevée (24h),True,False
7,P_00008,Meningiome,0.399052,TRES_FAIBLE,Elevée (24h),True,False
16,P_00017,Tumeur pituitaire,0.389591,TRES_FAIBLE,Elevée (24h),True,False
19,P_00020,Meningiome,0.348442,TRES_FAIBLE,Elevée (24h),True,False



Repartition des priorites:
priorite
Elevée (24h)     11
URGENTE (12h)     9
Name: count, dtype: int64

Alertes securite: 6
